# ── 0. 配置区 ──────────────────────────────────────────────

In [ ]:
# ── 0. 配置区 ──────────────────────────────────────────────
from pathlib import Path

# 输入：per-image CSV 所在根目录
PER_IMAGE_DIR = Path("./_eval_exports_per_images") / "sahi_null_v3"

# FiftyOne App URL（用于生成样本链接）
FO_BASE = "https://fiftyone.tianqiyao.men"

# 输出：HTML 报告根目录
OUT_ROOT = PER_IMAGE_DIR / "_prod_kiss_reports"

# 最多处理几个 CSV（None = 全部）
LIMIT: int | None = None

# Top N 错误表格
TOP_N = 20

# focus 列名
FOCUS_COL = "focus"

# ── 1. 初始化日志 ──────────────────────────────────────────

In [ ]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

# ── 2. 获取批量路径（Master 确认后继续）────────────────

In [ ]:
# ── Step 1：扫描所有待处理的 per-image CSV ─────────────────
# 输入：PER_IMAGE_DIR
# 输出：打印 CSV 列表供 Master 确认

from ty_plot_tools import find_per_image_csvs

logger.info("Step 1 开始：扫描 per-image CSV")

all_csvs = find_per_image_csvs(PER_IMAGE_DIR)
csvs = all_csvs[:LIMIT] if LIMIT is not None else all_csvs

print(f"共找到 {len(all_csvs)} 个 CSV，本次处理: {len(csvs)}")
for p in csvs:
    print(f'    "{p}",')

logger.info(f"Step 1 完成：待处理 {len(csvs)} 个 CSV")

# ── 3. Step 2：生成 HTML 报告 ───────────────────────────

In [ ]:
# ── Step 2：对每个 CSV 生成日报告 + 日概览 + 分钟概览 ──────
# 输入：csvs 中的每个 per-image CSV
# 输出：OUT_ROOT 下的 HTML 报告

from ty_plot_tools import (
    load_and_prepare_csv,
    write_image_level_day_html,
    write_daily_overview_html,
    write_minutely_swd_overview_html,
)

logger.info("Step 2 开始：生成 HTML 报告")

for csv_path in csvs:
    logger.info(f"[CSV] {csv_path}")

    df = load_and_prepare_csv(csv_path)
    if df is None:
        continue

    report_dir = OUT_ROOT / csv_path.parent.parent.name / csv_path.parent.name / csv_path.stem
    report_dir.mkdir(parents=True, exist_ok=True)

    day_to_file: dict[str, str] = {}

    for day, g in df.groupby("capture_date"):
        day = str(day)
        out_day_html = report_dir / f"image_level_{day}.html"
        try:
            write_image_level_day_html(
                df_day=g, out_html=out_day_html,
                title=f"{csv_path.stem} | {day}",
                top_n=TOP_N, focus_col=FOCUS_COL,
            )
            day_to_file[day] = out_day_html.name
        except Exception as e:
            logger.error(f"生成日报告失败 {out_day_html}: {e}")

    try:
        write_daily_overview_html(
            df=df, out_html=report_dir / "daily_overview.html",
            day_to_file=day_to_file,
            title=f"{csv_path.stem} | Daily overview",
        )
    except Exception as e:
        logger.error(f"生成日概览失败: {e}")

    try:
        write_minutely_swd_overview_html(
            df=df, out_html=report_dir / "MINUTELY_overview.html",
            title=f"{csv_path.stem} | MINUTELY SWD count",
            focus_col=FOCUS_COL,
        )
    except Exception as e:
        logger.error(f"生成分钟概览失败: {e}")

    logger.info(f"[DONE] report_dir = {report_dir}")

logger.info("Step 2 完成：所有报告生成结束")

# ── 4. 验证：检查输出目录 ───────────────────────────────

In [ ]:
# ── 验证：列出生成的报告文件 ────────────────────────────────
import os

logger.info(f"验证：OUT_ROOT={OUT_ROOT}")
html_files = list(OUT_ROOT.rglob("*.html"))
print(f"共生成 {len(html_files)} 个 HTML 文件")
for f in html_files[:10]:
    print(f"  {f}")